<a href="https://colab.research.google.com/github/Ololade117/Iroko/blob/main/Iroko_bot_finetuned3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install --upgrade --force-reinstall pyarrow
print("PyArrow reinstalled. Please restart the Colab runtime (Runtime -> Restart runtime) and then re-run all cells.")

Data Cleaning and Preprocessing

In [ ]:
import pandas as pd

# Define the URLs for the datasets
train_url = "https://raw.githubusercontent.com/Ololade117/Iroko/main/final_dataset/final_train.jsonl"
test_url = "https://raw.githubusercontent.com/Ololade117/Iroko/main/final_dataset/final_test.jsonl"
validation_url = "https://raw.githubusercontent.com/Ololade117/Iroko/main/final_dataset/final_validation.jsonl"

# Load each dataset into a pandas DataFrame
df_train = pd.read_json(train_url, lines=True)
df_test = pd.read_json(test_url, lines=True)
df_validation = pd.read_json(validation_url, lines=True)

# Concatenate the DataFrames into a single DataFrame
df_combined = pd.concat([df_train, df_test, df_validation], ignore_index=True)

# Display the first 5 rows of the combined DataFrame
display(df_combined.head())

In [ ]:
import re

# Initialize an empty list to hold processed text for each row
processed_texts = []

# Iterate through each row of the DataFrame
for _, row in df_combined.iterrows():
    # Concatenate the 'instruction', 'input', and 'output' columns for the current row
    # Convert each value to string to handle potential non-string data types
    row_text = f"{str(row['instruction'])} {str(row['input'])} {str(row['output'])}"
    processed_texts.append(row_text)

# Join all the row texts into one large string
large_data_string = " ".join(processed_texts)

# Define the words to be removed (case-insensitively, and as whole words)
words_to_remove = ["instruction", "input", "output", "index"]

# Preprocess the large string by removing the specified words
# Using regex for whole word replacement to be more precise and avoid partial matches
# Replacing with a space to prevent unintended word concatenation and then normalizing spaces
for word in words_to_remove:
    # Create a regex pattern for the whole word, case-insensitive
    pattern = r'\\b' + re.escape(word) + r'\\b'
    large_data_string = re.sub(pattern, ' ', large_data_string, flags=re.IGNORECASE)

# Normalize spaces: replace multiple spaces with a single space
# And strip leading/trailing whitespace
preprocessed_string = re.sub(r'\\s+', ' ', large_data_string).strip()

# Print information about the resulting string
print(f"Length of the preprocessed string: {len(preprocessed_string)}")
print("\nFirst 1000 characters of the preprocessed string:")
print(preprocessed_string[:1000])


In [ ]:
%%capture
# Force upgrade bitsandbytes and other key libraries to avoid quantization errors
!pip install -U bitsandbytes>=0.46.1
!pip install -U accelerate peft transformers trl datasets

In [ ]:
from huggingface_hub import login

# You will be prompted to enter your Hugging Face token
login()

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset


# ============================================================
# 1. Load Tokenizer
# ============================================================

model_id = "google/gemma-4-E4B-it"

print(f"Loading tokenizer for {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=False,
)

tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# 2. Convert DataFrame to Hugging Face Dataset
# ============================================================

print("Converting DataFrame to Dataset...")

raw_dataset = Dataset.from_pandas(df_combined)

# ============================================================
# 3. Format Each Example Using Gemma Chat Template
# ============================================================

def formatting_func(examples):
    texts = []

    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    for instruction, input_text, response in zip(
        instructions,
        inputs,
        outputs,
    ):

        instruction = instruction or ""
        input_text = input_text or ""
        response = response or ""

        if input_text.strip():
            prompt = f"{input_text}\n\n{instruction}"
        else:
            prompt = instruction

        messages = [
            {
                "role": "user",
                "content": prompt,
            },
            {
                "role": "assistant",
                "content": response,
            },
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)

    return {"text": texts}


print("Formatting dataset...")

formatted_dataset = raw_dataset.map(
    formatting_func,
    batched=True,
    remove_columns=raw_dataset.column_names,
)

# ============================================================
# 4. Train / Validation Split
# ============================================================

dataset = formatted_dataset.train_test_split(
    test_size=0.05,
    seed=42,
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# ============================================================
# 5. Tokenization
# ============================================================

MAX_LENGTH = 512


def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    # Explicitly create labels for causal LM
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


print("Tokenizing training dataset...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
)

print("Tokenizing validation dataset...")

eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
)

# ============================================================
# 6. Keep Only Required Columns
# ============================================================

required_columns = [
    "input_ids",
    "attention_mask",
    "labels",
]

train_dataset = train_dataset.remove_columns(
    [
        col
        for col in train_dataset.column_names
        if col not in required_columns
    ]
)

eval_dataset = eval_dataset.remove_columns(
    [
        col
        for col in eval_dataset.column_names
        if col not in required_columns
    ]
)

# ============================================================
# 7. Preview
# ============================================================

print(f"\nTraining examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")

print("\nExample:")

print(tokenizer.decode(train_dataset[0]["input_ids"]))

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import re
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# --- Explicitly release any prior run's GPU objects before loading a fresh model ---
for var_name in ["trainer", "model"]:
    if var_name in dir():
        exec(f"del {var_name}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"Memory Allocated before load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Memory Reserved before load: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# 1. Base Configuration
model_id = "google/gemma-4-E2B-it"

# 2. 4-bit Loading Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model in clean 4-bit space ({model_id})...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
    max_memory={0: "13GiB", "cpu": "30GiB"},
    low_cpu_mem_usage=True,
    dtype=torch.float16,
)

print(f"Memory Allocated after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def is_multimodal(name):
    return "vision_tower" in name or "audio_tower" in name

frozen_count = 0
for name, param in model.named_parameters():
    if is_multimodal(name):
        param.requires_grad = False
        frozen_count += 1
print(f"Explicitly froze {frozen_count} multimodal-tower parameters")

norm_upcast_count = 0
for name, module in model.named_modules():
    if is_multimodal(name):
        continue
    if isinstance(module, (nn.LayerNorm,)) or "RMSNorm" in type(module).__name__:
        for p in module.parameters(recurse=False):
            if p.dtype in (torch.float16, torch.bfloat16):
                p.data = p.data.to(torch.float32)
                norm_upcast_count += 1
print(f"Upcast {norm_upcast_count} language-model norm parameters to fp32")

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

gc.collect()
torch.cuda.empty_cache()
print(f"Memory Allocated after norm upcast: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def unwrap_clippable_linear(model):
    replaced = 0
    for parent in list(model.modules()):
        for child_name, child in list(parent.named_children()):
            if type(child).__name__ == "Gemma4ClippableLinear":
                setattr(parent, child_name, child.linear)
                replaced += 1
    return replaced

n_unwrapped = unwrap_clippable_linear(model)
print(f"Unwrapped {n_unwrapped} Gemma4ClippableLinear layers")

gc.collect()
torch.cuda.empty_cache()

target_modules_regex = r"^(?!.*(vision_tower|audio_tower)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules_regex,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

leaked = [n for n, p in model.named_parameters() if is_multimodal(n) and p.requires_grad]
print(f"LoRA params leaked into multimodal towers: {len(leaked)} (should be 0)")

trainable_dtypes = {p.dtype for n, p in model.named_parameters() if p.requires_grad}
print(f"Trainable parameter dtypes: {trainable_dtypes}")

# 4. Training Arguments — checkpointing + validation loss tracking + early stopping
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # --- Validation loss, tracked alongside training loss ---
    eval_strategy="steps",
    eval_steps=100,

    # --- Checkpointing, so a dropped session doesn't lose progress ---
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,  # keep only the 3 most recent checkpoints, to save disk

    # --- Auto-restore the best checkpoint by validation loss when training ends ---
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
)

print(f"Memory Allocated before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Memory Reserved before training: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# 5. Set up SFTTrainer — eval_dataset back in, plus early stopping on validation loss
print("\nSetting up SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_arguments,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# 6. Train — resumes automatically from the latest checkpoint if one exists
print("\nStarting fine-tuning...")
import os as _os
resume = bool(_os.path.isdir("./results") and any(_os.scandir("./results")))
trainer.train(resume_from_checkpoint=resume)

# 7. Save the fine-tuned model (this will be the BEST checkpoint by eval_loss, not necessarily the last)
output_dir = "./gemma-4-e4b-finetuned-mentalhealth-iroko"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned model saved to {output_dir}")

In [ ]:
# Define your Hugging Face repository name
repo_id = "gemma-4-e2b-iroko-mentalhealth-finetuned2"

print(f"Pushing model to Hugging Face Hub: {repo_id}...")

# Push the LoRA adapters
trainer.model.push_to_hub(repo_id)

# Push the tokenizer
tokenizer.push_to_hub(repo_id)

print("Model successfully pushed to the Hub!")

In [ ]:
import torch

model.eval()

def generate_response(prompt, max_new_tokens=200, temperature=0.7, top_p=0.9):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(
        output[0][input_len:], skip_special_tokens=True
    )
    return response.strip()

test_prompts = [
    "I've been feeling really overwhelmed with school and I don't know how to cope.",
    "I had a huge fight with my best friend and I don't know if we're okay anymore.",
    "I can't sleep because I keep thinking about everything that could go wrong.",
    "How do I know if what I'm feeling is normal stress or something more serious?",
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*70}")
    print(f"PROMPT {i}: {prompt}")
    print('='*70)
    response = generate_response(prompt)
    print(response)

model.train()

In [ ]:
import gradio as gr
import torch

model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = True  # speeds up generation

def chat_fn(message, history):
    # Build the full conversation from Gradio's history format
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(
        output[0][input_len:], skip_special_tokens=True
    )
    return response.strip()

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Iroko — Mental Health Support Chatbot",
    description="Fine-tuned Gemma 4 E2B model. This is a research/testing interface, not a substitute for professional mental health care.",
    examples=[
        "I've been feeling really overwhelmed with school and I don't know how to cope.",
        "I had a huge fight with my best friend and I don't know if we're okay anymore.",
        "I can't sleep because I keep thinking about everything that could go wrong.",
    ],
)

demo.launch(share=True, debug=True)